# AMP Finder AI — Explainable Random Forest Baseline

## Goal

Train one understandable model on fixed biological features, choose its decision threshold using validation data, evaluate it once on held-out test data, and save a versioned artifact for the Streamlit app.

## Setup

Run the biological-understanding notebook first. This notebook falls back to synthetic data only for a technical execution check.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src/amp_finder").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the AMP Finder AI project.")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
(PROJECT_ROOT / "outputs/figures").mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve

from amp_finder.modeling import save_artifact, train_feature_random_forest

sns.set_theme(style="whitegrid", context="notebook")
real_dataset_path = PROJECT_ROOT / "data/processed/amp_dataset.csv"
demo_dataset_path = PROJECT_ROOT / "data/demo/demo_sequences.csv"
dataset_path = real_dataset_path if real_dataset_path.exists() else demo_dataset_path
dataset = pd.read_csv(dataset_path)
is_demo = dataset_path == demo_dataset_path
print("Dataset:", dataset_path)
print("Status:", "SYNTHETIC UI DEMO — NOT SCIENTIFIC EVIDENCE" if is_demo else "Prepared APD/UniProt demonstration data")
dataset.groupby(["split", "label"]).size().unstack(fill_value=0)

## Steps

### 1. Fit on train and select the threshold on validation

In [ ]:
status = "synthetic_ui_demo_only" if is_demo else "real_data_demonstration"
artifact, predictions = train_feature_random_forest(
    dataset,
    random_state=42,
    scientific_status=status,
    dataset_notes=str(dataset_path),
)
print("Selected validation threshold:", round(artifact["threshold"], 3))
pd.DataFrame(
    [artifact["metadata"]["validation_metrics"], artifact["metadata"]["test_metrics"]],
    index=["validation", "held-out test"],
)[["n", "roc_auc", "average_precision", "balanced_accuracy", "mcc", "f1", "sensitivity_recall", "specificity", "brier_score"]].round(3)

### 2. Inspect held-out discrimination and errors

In [ ]:
test_predictions = predictions.query("split == 'test'").copy()
test_labels = test_predictions["label"].to_numpy()
test_scores = test_predictions["score"].to_numpy()
test_calls = test_predictions["prediction"].to_numpy()

fpr, tpr, _ = roc_curve(test_labels, test_scores)
precision, recall, _ = precision_recall_curve(test_labels, test_scores)
matrix = confusion_matrix(test_labels, test_calls, labels=[0, 1])

figure, axes = plt.subplots(1, 3, figsize=(15, 4.4))
axes[0].plot(fpr, tpr, color="#2F6BFF", linewidth=2)
axes[0].plot([0, 1], [0, 1], color="#98A2B3", linestyle="--")
axes[0].set(title="Held-out ROC curve", xlabel="False-positive rate", ylabel="True-positive rate")

axes[1].plot(recall, precision, color="#D59B2D", linewidth=2)
axes[1].set(title="Held-out precision–recall curve", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))

sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2])
axes[2].set(title="Held-out confusion matrix", xlabel="Predicted label", ylabel="True label")
axes[2].set_xticklabels(["Putative non-AMP", "AMP"], rotation=20)
axes[2].set_yticklabels(["Putative non-AMP", "AMP"], rotation=0)
figure.tight_layout()
evaluation_path = PROJECT_ROOT / "outputs/figures/02_baseline_evaluation.png"
figure.savefig(evaluation_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", evaluation_path)

### 3. Inspect global feature importance

In [ ]:
importance = (
    pd.DataFrame(artifact["feature_importance"].items(), columns=["feature", "importance"])
    .head(12)
    .sort_values("importance")
)
figure, axis = plt.subplots(figsize=(8.5, 6))
axis.barh(importance["feature"], importance["importance"], color="#2F6BFF")
axis.set_title("Random Forest global feature importance")
axis.set_xlabel("Impurity importance")
axis.set_ylabel("")
figure.tight_layout()
importance_path = PROJECT_ROOT / "outputs/figures/02_feature_importance.png"
figure.savefig(importance_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", importance_path)

### 4. Save the app artifact and predictions

In [ ]:
model_path = PROJECT_ROOT / ("models/notebook_demo_baseline_rf.joblib" if is_demo else "models/baseline_rf.joblib")
prediction_path = PROJECT_ROOT / ("outputs/notebook_demo_predictions.csv" if is_demo else "outputs/baseline_predictions.csv")
save_artifact(artifact, model_path)
predictions.to_csv(prediction_path, index=False)
print("Saved model:", model_path)
print("Saved predictions:", prediction_path)

## Checks

In [ ]:
assert set(predictions["split"]) == {"train", "validation", "test"}
assert predictions["score"].between(0, 1).all()
assert 0 < artifact["threshold"] < 1
assert artifact["metadata"]["test_metrics"]["n"] == len(test_predictions)
assert not dataset["sequence"].duplicated().any()
assert (dataset.groupby("split_group")["split"].nunique() == 1).all()
print("Training, prediction-range, held-out-count, duplicate, and split-group checks passed.")
if is_demo:
    print("Do not share these toy metrics. Rerun after creating data/processed/amp_dataset.csv.")

## Next Steps

1. Replace synthetic data with the prepared APD/UniProt dataset if necessary.
2. Run the ESM-2 notebook on the identical split assignments.
3. Compare held-out metrics and failure cases—not only the highest number.
4. Launch `streamlit run app.py` and confirm the real-data artifact is selected.